# 02a — Tariff Pricer (owns its calculation)
This notebook IS the tariff method: the CALC cell below computes the premium and registers it. `pricing.py` only connects (`quote`/`api_quote`). Hub `02_pricing.ipynb` and `03_main.ipynb` replay this exact cell via the loader — one source of truth.

In [ ]:
import copy, os, sys
ROOT = os.path.abspath('')
if ROOT not in sys.path: sys.path.insert(0, ROOT)

import pandas as pd
from voltvision import CFG, SCEN, N_YEARS, RateCard, describe_pricer, api_quote, gen, simulate
from voltvision import io
REGIME = 'tariff'


In [ ]:
# CALC — owned by this notebook. Edit the math here; the connector only calls it.
# The simulated book carries RISK ATTRIBUTES only; the tariff base premium
# (Schedule of Motor Tariff 2015) is computed HERE from those attributes:
#   Comp = first-RM1,000 rate + PER_EXTRA per extra thousand of sum assured
#   TPFT = 75% of Comp; TPO = flat table rate (SA slice added at pricing time)
import numpy as np
from voltvision import loading
from voltvision.config import BANDS, BASIC_COMP, BASIC_TPO, PER_EXTRA
from voltvision.pricing import register_pricer

TARIFF_INFO = {
    'label': 'Tariff',
    'params': ['sst', 'tpo_sa_pct', 'tpo_loading', 'risk_step'],
    'formula': ('BASIC(attributes) x loading x (1-NCD) x risk_step^flags x (1+sst); '
                'TPO = (BASIC + tpo_sa_pct x SA) x tpo_loading x risk_step^flags x (1+sst)'),
}


def basic_premium(book):
    # Tariff base premium per policy, straight from the 2015 schedule tables.
    bi = {b: i for i, b in enumerate(BANDS)}
    region = book['REGION'].values
    eng = book['ENGINE_CAPACITY'].values
    sa = book['SUM_ASSURED'].values
    comp = np.array([BASIC_COMP[r][bi[e]] for r, e in zip(region, eng)])
    extra = np.array([PER_EXTRA[r] for r in region])
    comp_basic = comp + extra * np.ceil(np.maximum(0, sa - 1000) / 1000)
    tpo = np.array([BASIC_TPO[r][bi[e]] for r, e in zip(region, eng)])
    cov = book['COVERAGE_TYPE'].values
    basic = np.where(cov == 'Comprehensive', comp_basic,
                     np.where(cov == 'TPFT', np.round(0.75 * comp_basic, 2), tpo))
    return np.round(basic, 2)


def price_tariff(book, card):
    o = book.copy()
    base = basic_premium(o)
    ncd = 1 - o['NCD_LEVEL'].values
    risk = card.risk_step ** (o['FLOOD_RISK'].values.astype(int) + o['THEFT_RISK'].values.astype(int))
    prem = base * loading(o) * ncd * risk * (1 + card.sst)
    t = o['COVERAGE_TYPE'].values == 'TPO'
    prem[t] = ((base[t] + card.tpo_sa_pct * o['SUM_ASSURED'].values[t])
               * card.tpo_loading * risk[t] * (1 + card.sst))
    return o.assign(FINAL_PREMIUM_SST=prem.round(2))


register_pricer('tariff', price_tariff, TARIFF_INFO)
print('tariff calc registered')


## Rule sheet (`describe_pricer('tariff')`)
| Piece | Rule |
|---|---|
| Comp / TPFT | `BASIC × loading × (1−NCD) × risk_step^flags × (1+sst)` |
| TPO | `(BASIC + tpo_sa_pct × SA) × tpo_loading × risk_step^flags × (1+sst)` — no cover loading, no NCD |
| `BASIC` | Schedule of Motor Tariff 2015 computed IN THIS notebook from book attributes: graduated Comp, TPFT = 0.75 × Comp, TPO flat by band/region |
| `loading` | driver band (1.20 / 1.05 / 1.00 / 1.05) × (1 + 0.03 × min(CAR_AGE, 10)) |
| `flags` | FLOOD_RISK + THEFT_RISK count |
| Live params | `sst`, `tpo_sa_pct`, `tpo_loading`, `risk_step` — edit the card cell |

In [ ]:
print(describe_pricer(REGIME))
card = RateCard.from_cfg(CFG)
# --- tweak here, e.g.: ---
# card.tpo_loading = 1.50
# card.sst = 0.10
display(pd.DataFrame(vars(card).items(), columns=['field', 'value']))


## Request (simulated book in)

In [ ]:
SC = "MIX"
try:
    book = io.load_sim(SC)
    print(f"loaded shared sim_{SC}: {len(book)} rows")
except FileNotFoundError:
    print('shared book missing — quick inline sim')
    c = copy.deepcopy(CFG)
    c['n'] = 2000
    book = simulate(gen(c, SCEN[SC], c['seed']), c, SCEN[SC],
                    seed=c['seed'], n_years=N_YEARS, verbose=False)


## Response (API format: regime + label + card + metrics + priced book)

In [ ]:
resp = api_quote(book, REGIME, card)
print('regime:', resp['regime'], '|', resp['label'])
display(pd.DataFrame([resp['metrics']]))
display(resp['book'][['POLID', 'COVERAGE_TYPE', 'VEHICLE_TYPE', 'CLAIM_COUNT',
    'CLAIM_AMOUNT', 'FINAL_PREMIUM_SST']].head())
p = io.save_priced(SC, resp['regime'], resp['book'])
print('saved ->', p)


## Notes
- TPO leg prices below expected cost (TPO LR >100%) — move `tpo_sa_pct` / `tpo_loading` above when approved.
- Siblings: `02b_glm.ipynb`, `02c_telem.ipynb` (same request/response shape).